# Morphological Feature Extraction

**Purpose:** Extracts morphological node statistics (axon, basal dendrite, and apical dendrite
lengths and standard deviations) from eCREST `.json` reconstruction files for each cell type.
These features are used downstream for cell type classification.

**Inputs:**
- `EM_data_published/reconstructions_published/` — eCREST `.json` reconstruction files
- `EM_data_published/data_processed_published/df_type_auto_typed.csv` — cell type assignments

**Outputs:**
- `EM_data_published/data_processed_published/morphology_cat/df_nodes_*.csv` files (one per cell type)

**Used by:** `CellTyping.ipynb`

**Launch requirement:** Run `jupyter lab` from `efish_em/Notebooks_Jupyter/`

# setup

In [1]:
############################################################################################################################
# Get the latest CREST files for each ID within the target folder (dirname)

import cloudvolume
from cloudvolume.exceptions import EmptyFileException
# import google.colab.auth
from google.cloud import bigquery

from pathlib import Path
import json
from sqlite3 import connect as sqlite3_connect
from sqlite3 import DatabaseError
from igraph import Graph as ig_Graph
from igraph import plot as ig_plot
from scipy.spatial.distance import cdist
from random import choice as random_choice
from itertools import combinations
from numpy import array, unravel_index, argmin, mean
import random
import numpy as np
from copy import deepcopy
import itertools
from datetime import datetime
from time import time
import neuroglancer
from webbrowser import open as wb_open
from webbrowser import open_new as wb_open_new
import pandas as pd
import seaborn as sns
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from tqdm import tqdm
from datetime import datetime

In [2]:
'''
conda install conda-forge::google-cloud-sdk

Then, launch jupyter lab 

In a code cell, run bash command <!gcloud auth login > (https://cloud.google.com/sdk/gcloud/reference/auth/login)
    a browser tab should open up

RESULT:
You are now logged in as [kperky@gmail.com].
Your current project is [lcht-goog-connectomics].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID
'''
!gcloud auth login 

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=CZuO6dKoe7nl1g9OZNtG7OaUBzAmBc&access_type=offline&code_challenge=N4StEIl1mloVF3wxtWQsoa8PBmHf8SDciHFjoR-Ih7s&code_challenge_method=S256


You are now logged in as [kperky@gmail.com].
Your current project is [lcht-goog-connectomics].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID


In [2]:
import sys
from pathlib import Path
DIR_ROOT = Path.cwd().parent / 'efish_em'
if str(DIR_ROOT) not in sys.path:
    sys.path.insert(0, str(DIR_ROOT))

In [3]:
# from eCREST_cli_beta import ecrest, import_settings
from eCREST_cli import ecrest, import_settings, get_cell_filepaths
import AnalysisCode as efish

In [4]:
# Path to eCREST settings file — update this to your local eCREST settings location
# Example: eCREST-local-files/ should be a sibling of the efish_em/ directory
path_to_settings_json = Path.cwd().parent.parent / 'eCREST-local-files' / 'settings_dict.json'
settings_dict = efish.import_settings(path_to_settings_json)

vx_sizes = [16,16,30]

db_cursors = sqlite3_connect(settings_dict['db_path'], check_same_thread=False).cursor()

a = ', '.join(['base_address'])

db_cursors.execute(f'''SELECT {a} FROM addresses_table LIMIT 1''')

[base_seg] = db_cursors.fetchall()[0]

mesh_seg = 'precomputed://gs+ngauth+https://ngauth-goog.appspot.com/fish-ell/roi450um_seg32fb16fb_220930'

# get molecular layer plane fit function

In [5]:
neuroglancer_path = Path(settings_dict['save_dir']).parent.parent / 'blender/soma_locations/layer-molecular_annotation.json'

with open(Path(neuroglancer_path), 'r') as myfile: # 'p' is the dirpath and 'f' is the filename from the created 'd' dictionary
    neuroglancer_data = json.load(myfile)

set([item['name'] for item in neuroglancer_data['layers'] if item['type']=='annotation'])

nl_ = 'molecular'
neuroglancer_layer = next((item for item in neuroglancer_data['layers'] if item["name"] == nl_), None)
voxel_sizes = [16,16,30]

vertices = [[p['point'][i]*voxel_sizes[i] for i in range(3)] for p in neuroglancer_layer['annotations']] #[p['point'] for p in neuroglancer_layer['annotations']]#

x_pts = [p[0] for p in vertices]
y_pts = [p[1] for p in vertices]
z_pts = [p[2] for p in vertices]

# Perform curve fitting
popt, pcov = curve_fit(efish.func_planar_curve, (x_pts, z_pts), y_pts)

# Print optimized parameters
print(popt)


[ 2.71956920e+05 -5.43115077e-02 -1.87026179e-01 -3.46153667e-07
  2.31048373e-06  9.59242290e-13 -1.51595014e-11  6.68290149e-07]


# load file list

In [6]:
dirpath = Path(settings_dict['save_dir'])

nodefiles = efish.get_cell_filepaths(dirpath)

# cell types all cells


## From saved file

In [7]:
df_type = pd.read_csv(dirpath / 'metadata/df_type_auto_typed.csv')

In [10]:
cell_type = dict(zip(df_type['id'].values, df_type['cell_type'].values))

## new from directory

In [11]:
nodefiles = efish.get_cell_filepaths(Path(settings_dict['save_dir']))

In [ ]:
cell_type = {}
not_typed = []
for x,f in nodefiles.items():
    cell = ecrest(settings_dict,filepath = f,launch_viewer=False)
    cell_type[x] = cell.get_ctype('manual')
    if (cell.get_ctype('manual') == []) | (cell.get_ctype('manual') == ''):
        cell_type[x]=''
        not_typed.append(x)# print(f'cell {x} is not cell-typed in json')

print('the following cells are not typed in the main network')
print(not_typed)


the following cells are not typed in the main network
[]


In [ ]:
df_type = pd.DataFrame(cell_type.items(),columns = ['id','cell_type'])

In [ ]:
df_type.loc[df_type['cell_type'].isin(['dml']),'cell_type']='mli'
df_type.loc[df_type['cell_type'].isin(['grc-d']),'cell_type']='grc'
df_type.loc[df_type['cell_type'].isin(['grc-s']),'cell_type']='smpl'

In [ ]:
df_type.to_csv(dirpath / 'metadata/df_type.csv')

# Get cell segment skeleton node locations

## load cloudvolume and bigquery


In [8]:
efish_cloudvolume = cloudvolume.CloudVolume('gs://fish-ell/roi450um_seg32fb16fb_220930', progress=False)

In [9]:

bigquery_client = bigquery.Client(project='lcht-goog-connectomics')

/Users/kperks/opt/anaconda3/envs/ell-kimimaro/lib/python3.8/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


## which cells and segments

In [10]:
cutoff_date = datetime.strptime('2025-03-31', '%Y-%m-%d')

# Filter keys with filenames more recent than cutoff_date
recent_cell_ids = [
    int(cell_id)
    for cell_id, filename in nodefiles.items()
    if datetime.strptime(nodefiles[str(cell_id)].name.split('__')[1].split(' ')[0],'%Y-%m-%d') > cutoff_date
]


In [37]:
cell_to_skel = df_type[df_type['cell_type'].isin(['sg2','mg2'])]['id'].values #['mg1','mg2','lg','lf']

In [38]:
cell_to_skel = list(set(cell_to_skel).intersection(set(recent_cell_ids)))

In [48]:
segs_toanalyze = {}
dtype = 'axon'
with tqdm(total=len(cell_to_skel)) as pbar:
  for focal_cell_id in cell_to_skel:
      pbar.update(1)
      cell_data = efish.load_ecrest_celldata(nodefiles[str(focal_cell_id)])
      mesh_ids = cell_data['base_segments'][dtype]
      segs_toanalyze[focal_cell_id] = mesh_ids

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 29.07it/s]


## create and save dataframe of segment skeleton info


In [49]:
suff='ax'  # 'ad' 'bd' 'ax'
dfsavepath = Path(settings_dict['save_dir']).parent / 'meta_analysis/morphology_cat/'
dffilename = f'df_nodes_{suff}_type2_updates.csv'


In [50]:
reformatted_dfs = []
for focal_cell_id,mesh_ids in segs_toanalyze.items():
    
    all_verts = []
    with tqdm(total=len(mesh_ids)) as pbar:
        for segment_id in mesh_ids:
            pbar.update(1)
            try:
              skel = efish_cloudvolume.skeleton.get(int(segment_id));
            
              # Iterate over each vertex in the skeleton
              for v in skel.vertices:
            
                  #zero y from molecular layer plane
                  yoffset = efish.func_planar_curve((v[0], v[2]), *popt)
                  y_adj = v[1] - yoffset
                  v = [int(v[0]),int(y_adj),int(v[2])]
            
                  # Append the vertex coordinates to the list
                  all_verts.append(list(v))
            
            except EmptyFileException: # if the segment is too small it doesn't have a skeleton
              # print(f"EmptyFileException: segid {segment_id} is missing. using this segment's location.")
              try:
                  QUERY = """
                  SELECT
                      cast(objects.id as INT64) as seg_id,
                      sample_voxel.x as x,
                      sample_voxel.y as y,
                      sample_voxel.z as z,
                  FROM
                      `lcht-goog-connectomics.ell_roi450um_seg32fb16fb_220930.objinfo` as objects
                  WHERE objects.id in {}
                  """.format('('+str(segment_id)+')')
            
                  df_seg = bigquery_client.query(QUERY).to_dataframe()
                  #zero y from molecular layer plane
                  yoffset = efish.func_planar_curve((df_seg['x'].to_list()[0], df_seg['z'].to_list()[0]), *popt)
                  y_adj = df_seg['y'].to_list()[0] - yoffset
                  v = [df_seg['x'].to_list()[0],int(y_adj),df_seg['z'].to_list()[0]]
                  all_verts.append(list(v))
                  continue
            
              except IndexError:
                  continue
                  
    # all_verts = [v[0] for v in all_verts]
    df = pd.DataFrame(all_verts, columns=['x', 'y', 'z'])
    r = df.describe()
      
    df_flat = r.T.stack().to_frame().T
    df_flat.columns = [f"{suff}_{stat}_{col}" for col, stat in df_flat.columns]
    df_flat['id'] = focal_cell_id

    reformatted_dfs.append(df_flat)

final_df = pd.concat(reformatted_dfs, ignore_index=True)
final_df.set_index('id',inplace=True)
final_df.to_csv(dfsavepath / dffilename)
# #################


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 78/78 [00:19<00:00,  4.02it/s]


## combine updated df with existing csv and resave

In [51]:
fname = '_'.join(dffilename.split('_')[0:-1])
existing_df = pd.read_csv(dfsavepath / f'{fname}.csv')

combined_df = (
    pd.concat([existing_df, final_df.reset_index()])
    .drop_duplicates(subset='id', keep='last')
    .reset_index(drop=True)
)

combined_df.to_csv(dfsavepath / f'{fname}.csv',index=False)